# Take-home group exercise: fine-tune on open WAXAL data in Colab (UNTESTED in Colab)

**Status (20 Oct 2026): not run in Colab.** Its code cells were run once on a facilitator GPU (RTX 5080, torch 2.8, transformers 4.57.6) for 10 steps, with local copies of the same two WAXAL files (same SHA-256); the saved model loaded in `asr.engine.WhisperEngine` and transcribed `good.wav`. The facilitators' full runs used the same recipe (`_gpu/finetune_waxal.py`). Colab timing and the T4 fp16 path are untested.

Goal: fine-tune the lab champion (Whisper tiny) on **one shard of WAXAL Amharic train** (`google/WaxalNLP`, CC-BY-SA-4.0: 1,841 clips, about 8.9 h of image-prompted speech, 186 speakers), with
1. the data pinned by dataset revision and SHA-256,
2. a **speaker** leakage check against the WAXAL test speakers (episode 2: group by speaker),
3. every parameter, the data hash and the seed in MLflow (REP-01),

then bring the model back and judge it with `scripts/promote.py` on the lab's eval sets.

Runtime: Colab GPU (T4), roughly 25-35 min for 400 steps plus a 0.5 GB download (estimate). Upload nothing personal.
Licence: WAXAL Amharic is CC-BY-SA-4.0. Keep the attribution. Whether ShareAlike reaches the fine-tuned weights is a legal question; ask before you publish a model.

In [ ]:
!pip -q install transformers==4.57.6 jiwer mlflow soundfile pyarrow

In [ ]:
import hashlib, io
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
REV = '5f4d8ca24f2b9d168b2ee545f1febaaff4b40580'   # google/WaxalNLP revision used by the facilitators
TRAIN_SHA = 'd38389c4e22924369b940e927ab7a79212d12c13975265c0add327ede3aa622d'
def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()
train_f = hf_hub_download('google/WaxalNLP', 'data/ASR/amh/amh-train-00000.parquet', repo_type='dataset', revision=REV)
train_sha = sha256(train_f)
print('train shard sha256', train_sha, 'OK' if train_sha == TRAIN_SHA else 'DIFFERENT FROM THE FACILITATORS')
# Leakage check (episode 2): are any training speakers in the test split? Reads only the speaker_id column.
test_f = hf_hub_download('google/WaxalNLP', 'data/ASR/amh/amh-test-00000.parquet', repo_type='dataset', revision=REV)
tr = pq.read_table(train_f, columns=['id', 'speaker_id', 'gender', 'transcription']).to_pandas()
te_spk = set(pq.read_table(test_f, columns=['speaker_id']).column('speaker_id').to_pylist())
leak = len(set(tr.speaker_id) & te_spk)
print('train clips', len(tr), 'speakers', tr.speaker_id.nunique(), tr.gender.value_counts().to_dict())
print('train speakers also in test:', leak)   # must be 0

In [ ]:
import numpy as np, soundfile as sf
from math import gcd
from scipy.signal import resample_poly
from transformers import WhisperProcessor
mid = 'b1n1yam/shook-tiny-amharic-600hr'
proc = WhisperProcessor.from_pretrained(mid)
tok = proc.tokenizer
tok.set_prefix_tokens(language='amharic', task='transcribe', predict_timestamps=False)
audio = pq.read_table(train_f, columns=['audio']).column('audio').to_pylist()
data = []
for row, a in zip(tr.itertuples(), audio):
    x, sr = sf.read(io.BytesIO(a['bytes']), dtype='float32')      # WAXAL audio: mp3, 48 kHz, stereo
    x = x.mean(axis=1) if x.ndim > 1 else x
    k = gcd(sr, 16000)
    x = resample_poly(x, 16000 // k, sr // k).astype('float32')
    ids = tok(row.transcription).input_ids    # <|startoftranscript|><|am|><|transcribe|><|notimestamps|> ... <|endoftext|>
    if 1 <= len(x) / 16000 <= 30 and len(ids) <= 440:     # Whisper: 30 s of audio, 448 decoder positions
        data.append((x, ids))
print('kept', len(data), 'of', len(tr), f'({sum(len(x) for x, _ in data) / 16000 / 3600:.1f} h)')

In [ ]:
import math, random, time, torch, mlflow
from transformers import WhisperForConditionalGeneration
P = dict(steps=400, lr=1e-5, batch_size=16, warmup=40, seed=0, max_label_tokens=440)
random.seed(P['seed']); np.random.seed(P['seed']); torch.manual_seed(P['seed'])
model = WhisperForConditionalGeneration.from_pretrained(mid).cuda()
model.config.use_cache = False
opt = torch.optim.AdamW(model.parameters(), lr=P['lr'])
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: min(1, (s + 1) / P['warmup']) * 0.5 * (1 + math.cos(math.pi * s / P['steps'])))
scaler = torch.amp.GradScaler('cuda')
mlflow.set_experiment('amharic-asr-finetune')
with mlflow.start_run(run_name='colab-tiny-waxal-shard0'):
    mlflow.log_params({**P, 'base_model': mid, 'dataset': 'google/WaxalNLP ASR/amh train-00000',
                       'dataset_revision': REV, 'train_sha256': train_sha, 'n_train': len(data), 'leak_speakers': leak})
    order = []
    model.train()
    t0 = time.time()
    for step in range(P['steps']):
        if len(order) < P['batch_size']:
            order = list(np.random.permutation(len(data)))
        batch = [data[order.pop()] for _ in range(P['batch_size'])]
        feats = proc.feature_extractor([x for x, _ in batch], sampling_rate=16000, return_tensors='pt').input_features.cuda()
        L = max(len(l) for _, l in batch)
        lab = torch.full((len(batch), L), -100)
        for j, (_, l) in enumerate(batch):
            lab[j, :len(l)] = torch.tensor(l)
        dec_in = lab[:, :-1].clone()
        dec_in[dec_in == -100] = tok.pad_token_id      # teacher forcing: predict token t+1 from tokens up to t
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(input_features=feats, decoder_input_ids=dec_in.cuda()).logits
        loss = torch.nn.functional.cross_entropy(logits.float().reshape(-1, logits.size(-1)),
                                                 lab[:, 1:].reshape(-1).cuda(), ignore_index=-100)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
        if (step + 1) % 25 == 0:
            mlflow.log_metric('train_loss', loss.item(), step=step + 1)
            print(step + 1, round(loss.item(), 4), f'{(time.time() - t0) / (step + 1):.2f} s/step')
    model.config.use_cache = True
    model.generation_config.forced_decoder_ids = None
    model.save_pretrained('tiny-waxal-ft'); proc.save_pretrained('tiny-waxal-ft')
    mlflow.log_param('weights_sha256', sha256('tiny-waxal-ft/model.safetensors'))

In [ ]:
!zip -qr tiny-waxal-ft.zip tiny-waxal-ft && ls -la tiny-waxal-ft.zip

Next: download `tiny-waxal-ft.zip`, unzip it on your laptop as `models/tiny-waxal-ft`, add
`myteam: {id: tiny-waxal-ft, path: models/tiny-waxal-ft}` under `models:` in `configs/params.yaml`, then:

```
python -m asr.evaluate --model myteam
python scripts/promote.py --candidate tiny-waxal-ft
```

Did the rule accept it? Which group changed most (phone channel, long clips)? Your model saw 9 h of one speech style from 186 speakers: what would you need before Gate 2? The facilitators' longer run (4 shards, 35 h) is described in `docs/AMHARIC_SPEECH_DATA.md`.